# Chapter 1 — The bottom line, and how to read the rest

Problem A is the single-hand blackjack decision (hit / stand / double / split / surrender), and it has
a known *reference* policy: basic strategy, the optimal play for this total-based abstraction (value, soft, upcard). That makes it a rare thing — a
reinforcement-learning problem where we already know the right answer for every state. So we can use
it as a **ruler**: train a neural agent, and measure precisely where and how it departs from the truth.

This chapter puts the result up front, then hands you the two tools you need for everything after: the
**two metrics** (and why they disagree), and the **decision-tree frame** (the lens that, by Chapter 7,
explains every finding).

### The chapters

1. **The bottom line, and how to read the rest** — the scoreboard, the agreement-vs-edge lens, the decision-tree frame. *(this chapter)*
2. **The diagnosis: the wild double** — the oscillating value on the terminal double, its two mechanisms, the Q-grid.
3. **Cornering the cause** — every fixable explanation (capacity, coverage, settling, learning-order, reward variance, the standard knobs) tested and ruled out — each leaving a usable method behind.
4. **The best-supported cause: representation** — where the errors live (the low-margin seam = the function-approximation floor), and encoding as a sharpness dial.
5. **Honesty and dynamics** — how we kept ourselves honest, and what more training actually buys (a plateau, not a climb — the peak you see is a transient).
6. **Completing the game** — adding split, then surrender, and what the full action set changes.
7. **What this actually was** — the synthesis: every finding from three axes, and why "the table wins" is the measured cost of model-free generality, not a defeat.

In [1]:
import sys; sys.path.insert(0, '.')
import statistics as st
import pandas as pd
from runs import load_runs, learning_curve, cell_categories, agreement_on, sample_counts, diff_cells, show, provenance, describe, OPTIMUM_PCT, EDGE_SE_PCT, tight_edge

df = load_runs(); dqn = df[df.method == 'dqn']

def back_half(path):
    a = [cp['agreement'] for cp in learning_curve(path) if cp.get('agreement') is not None]
    return st.mean(a[len(a) // 2:]) if a else float('nan')

# the network's canonical 240-cell grid (any no-split DQN run) — the common ground for scoring
net_grid = set(cell_categories(dqn[~dqn.with_splits].iloc[0].path))
# tabular Q-learner (no-split, 5M) scored on that same grid; naive DQN; the best DQN we reached
tab = df[(df.method == 'tabular') & (df.episodes == 5_000_000) & (~df.with_splits)]
naive = dqn[(dqn.hidden == '[64, 64]') & (dqn.episodes == 1_000_000) & (~dqn.swa) & (dqn.lr_schedule == 'constant')
            & (dqn.reward_baseline == 'none') & (dqn.double_after == 0) & (~dqn.with_splits)]
naive = naive[naive.edge_pct < 4.0]                       # drop one divergent-edge run (Ch.5)
best = dqn[(~dqn.with_splits) & dqn.has_model].assign(bh=lambda d: [back_half(p) for p in d.path]) \
          .sort_values('bh').iloc[-1]

# The reference these no-split agents could reach is the NO-SPLIT basic strategy. The per-run basic_edge_pct
# is ONE eval_seed=0, 200k-hand sample (SE ~0.26%/hand) that drew low (~0.09%), so it is NOT the optimum.
# The true 6-deck S17 3:2 basic edge is ~0.54% with the full action set (measured in-harness at 0.58%±0.10
# over 1.4M hands), and ~1.11% with no split/surrender — the proper reference for these no-split agents
# (see runs.OPTIMUM_PCT). Basic plays the FULL set; the no-split agents cannot, so 1.11% is their ceiling.
basic_edge = OPTIMUM_PCT['trimmed']

scoreboard = pd.DataFrame([
    {'method': 'tabular Q-learner', 'agreement': st.mean([agreement_on(p, net_grid) for p in tab.path]), 'edge': tight_edge('20260617-152831_seed42_c298e9c', tab.edge_pct.mean()), 'tuning': 'none'},
    {'method': 'naive DQN',         'agreement': naive.agreement.mean(),                                  'edge': naive.edge_pct.mean(), 'tuning': 'defaults'},
    {'method': 'best DQN',          'agreement': back_half(best.path),                                    'edge': tight_edge(best.run, best.edge_pct), 'tuning': 'full stack'},
    {'method': 'basic strategy (no-split optimum)', 'agreement': float('nan'),                            'edge': basic_edge,           'tuning': '—'},
])
show(scoreboard, pct=['agreement'], num=['edge'],
     caption='Problem A scoreboard — agreement = match to the table on the common 240-cell grid (exact, '
             'noiseless); edge = % of stake lost per hand, lower better. The optimum row is the NO-SPLIT basic '
             'strategy (~1.11%), the ceiling these no-split agents could reach; every learner edge is one 200k '
             'seed-0 eval (±0.26%/hand), so edge gaps under ~0.5% are within noise and should not be ranked.',
     source=('source — agreement is exact; edges: best DQN & tabular = 5M re-evals (reeval_results.json), naive = 200k (no weights):<br>'
             '&nbsp;&nbsp;tabular Q-learner — ' + provenance(tab).replace('source: ', '') + '<br>'
             '&nbsp;&nbsp;naive DQN — ' + provenance(naive).replace('source: ', '') + '<br>'
             '&nbsp;&nbsp;best DQN — ' + provenance(best).replace('source: ', '') + ' · scored on its back-half<br>'
             '&nbsp;&nbsp;no-split optimum — basic strategy, no split/surrender: literature ≈1.11%; in-harness full basic 0.58%±0.10 over 1.4M hands'))

method,agreement,edge,tuning
tabular Q-learner,92.8%,0.99,none
naive DQN,82.1%,1.94,defaults
best DQN,91.1%,1.08,full stack
basic strategy (no-split optimum),nan%,1.11,—


## 1.1 The headline

Two readings of the same three rows:

- On **agreement** (does the policy match the table cell-by-cell), the tabular learner leads (~0.93) and
  the best network trails by a couple of points (~0.91) — and it took the *full stack* of fixes to get
  there, against the table's *zero* tuning.
- On **edge** (money actually lost per hand), the best network and the table are **within noise of each
  other** (~1.08% vs ~0.99%, a gap inside the seed band) — neither "wins" on edge at this
  precision. The proper reference for these no-split agents is the **no-split basic strategy, ~1.11%**,
  and both learners sit within noise of it. (These are tight multi-million-hand re-evaluations, but seed-sensitive (~±0.15%),
  so read edge gaps under ~0.2% as within noise, not as exact rankings.)

Every edge number is **one** measurement — a tight multi-million-hand re-evaluation — but the eval is seed-sensitive, so it
carries a ~±0.15%/hand seed band (gaps under ~0.2% are not ranked); agreement, by contrast, is computed from the deterministic policy
and is **exact**. The only structural asymmetry is the action set — basic may split, the no-split agents
may not — which is itself part of the story (it is why their ceiling is ~1.11%, not the full game's ~0.54%).

So the honest verdict is **not** "the net failed." It's: *the table wins on exactness, simplicity, and
zero tuning — not on capability.* On edge the net is within noise of the table and of its own optimum;
hold onto the fact that "table wins on exactness" and "net is competent" are both true, because Chapter 7
pays it off.

One more thing the scoreboard hides: the two methods don't even evaluate the *same cells natively*. The
network has a learned answer at **every** canonical cell (it generalizes); the table only has the cells
it actually **visited**. We score both on their shared grid here, but that mismatch isn't a nuisance to
paper over — it *is* a finding (generalization vs coverage), and it returns in Chapter 7.

## 1.2 The two-metric lens (agreement vs edge)

Those two columns disagree on purpose, and keeping them apart is the single most useful habit for
reading everything that follows. Define them once:

- **Agreement** — fraction of the 240 cells where the agent's action equals basic strategy's. Every
  cell counts **equally**, common or rare. Computed from the deterministic policy, so it is **exact**.
- **Edge** — the house edge measured by playing many hands. Each mistake is weighted by **how often that
  cell occurs and how much it costs** — and even at millions of hands it carries a seed-sensitive ~±0.15%/hand band.

They diverge whenever the agent's mistakes are concentrated in cells that are *less frequent* or
*cheap* — then agreement drops but edge barely moves. That's exactly what the scoreboard shows (the net
trails on agreement while its edge stays within noise of the table's), and the cell below makes it
concrete: the net's disagreement cells are visited only about *half as often* as the cells it gets
right, so they carry less weight in money terms.

A *third* number is tempting and misleading: the agent's **self-gap** — how much better it *thinks* its
chosen action is than the alternative. Smaller self-gaps look like "more confident, so better," but they
don't track edge at all (Chapter 5 shows a run that shrank its self-gap while its edge stood still).
Trust agreement and edge; treat the self-gap as the agent grading its own homework.

In [2]:
# why its few errors barely move edge: its wrong cells are the lower-frequency ones
sc = sample_counts(best.path)
def freq(c): return sum(sc.get((c['player_value'], c['is_soft'], c['dealer_upcard'], a), 0) for a in ('hit', 'stand', 'double'))
cells = diff_cells(best.path)
dis = [c for c in cells if c['category'] == 'genuine_disagreement']
agr = [c for c in cells if c['category'] == 'agree']
freqcmp = pd.DataFrame([
    {'cell group': 'cells it gets RIGHT', 'count': len(agr), 'mean training visits': st.mean(freq(c) for c in agr)},
    {'cell group': 'cells it gets WRONG', 'count': len(dis), 'mean training visits': st.mean(freq(c) for c in dis)},
])
show(freqcmp, num=['mean training visits'],
     caption='Best DQN: its disagreements sit in the lower-frequency cells — lighter in edge, full weight in agreement.',
     source=provenance(best, role='best DQN'))

cell group,count,mean training visits
cells it gets RIGHT,223,26729.57
cells it gets WRONG,15,14445.00


## 1.3 The decision-tree frame (the spine)

Here is the lens for the whole document. **Basic strategy is a decision tree** — and your own one-page
rule list ("hard 17+ → stand; 11 → double; split aces and eights…") *is* that tree written out: a
stack of threshold rules, one action per region. A neural network is a *different kind* of partition of
the same space. The two differ along **three axes**, and every finding in this project is one of these
axes showing up:

| axis | the table (decision tree) | the network |
|---|---|---|
| **boundary** | a **hard** wall — you're on one side or the other | a **soft** ramp — it interpolates across |
| **cut direction** | **axis-aligned** — splits on one feature ("total ≥ 17") | **oblique** — splits on a weighted mix of features |
| **cells** | **independent** — each leaf is its own number | **shared** — adjacent cells share weights |

These three words — **hard/soft, axis-aligned/oblique, independent/shared** — are load-bearing. Read
ahead with them in hand:

- the **over-doubling** at boundaries (Ch. 2, 4) is *soft cuts where the tree has walls*;
- the **oscillation** of the double (Ch. 2) is *shared/coupled values where the tree has independent leaves*;
- the **encoding mattering** (Ch. 4) is *the smoothness prior fighting a hard, axis-aligned boundary*;
- the **capacity not mattering** (Ch. 3) is *the boundaries being representable all along*.

We introduce this as a *frame*, not a conclusion. Chapters 2–6 are the evidence; Chapter 7 returns here
and re-derives the entire project from these three axes — and from one last reframe: on a problem whose
model is *known*, computing the table is the right answer, and the network's gap is the measurable
**cost of pretending you don't know the model**. That's what this whole investigation actually measures.